# 확률론 3주차 과제 2 — Pascal-Fermat 상금 분배 문제

---

**과목**: 확률론  
**주차**: 3주차  
**주제**: 미완료 게임의 공평한 상금 분배 (Problem of Points)  

---

## 학습 목표

1. 확률론 탄생의 역사적 맥락을 이해한다
2. 결합확률과 동시확률의 개념을 학습한다
3. Pascal의 풀이(의사결정 트리)와 Fermat의 풀이(완전 열거)를 비교한다
4. 시뮬레이션을 통해 이론적 결과를 검증한다

## 역사적 배경

### 1654년, 확률론의 탄생

프랑스의 도박사 **슈발리에 드 메레(Chevalier de Méré)**가 수학자 **블레즈 파스칼(Blaise Pascal)**에게 다음과 같은 문제를 제시했다:

> 두 사람이 게임을 하다가 중단해야 할 때, 상금을 어떻게 나누는 것이 공평한가?

파스칼은 이 문제를 **피에르 드 페르마(Pierre de Fermat)**에게 편지로 보냈고, 두 수학자가 서신을 주고받으며 해결한 이 문제가 **현대 확률론의 시초**가 되었다.

이 문제가 중요한 이유는:
- **과거의 결과가 아닌 미래의 가능성**에 기반하여 가치를 산정하는 첫 번째 수학적 방법론이었다.
- 이는 현대의 **기대값(Expected Value)** 개념의 출발점이다.
- 보험, 금융, 의사결정 이론의 수학적 기반을 제공했다.

## 문제 정의

### 게임 규칙

- **상금**: 1억 원 (총 상금)
- **참가자**: A와 B, 각 라운드에서 이길 확률 $p = 0.5$ (공정한 게임)
- **승리 조건**: 먼저 **6승**을 달성한 사람이 전체 상금을 가져감
- **현재 상황**: A = 5승, B = 3승에서 게임이 중단됨

### 핵심 질문

> **1억 원을 A와 B에게 어떻게 나누는 것이 공평한가?**

### 직관적 접근의 함정

| 잘못된 접근 | 분배 | 문제점 |
|------------|------|--------|
| 승수 비례 배분 | A: 5/8, B: 3/8 | 과거 결과만 반영, 미래 가능성 무시 |
| 남은 승수 비례 | A: 3/4, B: 1/4 | A는 1승만 필요, B는 3승 필요라는 비대칭 무시 |

**올바른 접근**: 앞으로의 남은 게임에서 각자가 **최종 승리할 확률**을 계산해야 한다.

## 주요 확률 개념

### 결합확률 (Joint Probability)

두 사건 $A$와 $B$가 **동시에** 일어날 확률:

$$P(A \cap B) = P(A \text{ 그리고 } B)$$

두 사건이 **독립(independent)**이면:

$$P(A \cap B) = P(A) \times P(B)$$

### 독립사건 (Independent Events)

한 사건의 결과가 다른 사건의 확률에 영향을 미치지 않을 때 두 사건은 독립이다:

$$P(A | B) = P(A)$$

각 라운드의 승패는 **독립**이므로, 여러 라운드의 결합확률은 각 라운드 확률의 **곱**이다.

### 배반사건 (Mutually Exclusive Events)

두 사건이 동시에 일어날 수 없을 때:

$$P(A \cup B) = P(A) + P(B) \quad (\text{if } A \cap B = \emptyset)$$

A의 최종 승리와 B의 최종 승리는 **배반사건**이므로, 두 확률의 합은 1이다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import binom, bernoulli, norm, uniform, expon
from itertools import product
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

---

## 풀이 1: Pascal의 접근 — 의사결정 트리 (Decision Tree)

### 핵심 아이디어

현재 A=5승, B=3승이므로:
- A는 **1승만 더** 하면 승리
- B는 **3승을 더** 해야 승리

남은 라운드를 하나씩 진행하면서, 각 분기에서의 확률을 추적한다.

### 라운드별 분석

**라운드 1** (6번째 판):
- A가 이기면 → A 최종 승리 (6승 달성). 확률: $\frac{1}{2}$
- B가 이기면 → A=5, B=4. 계속 진행. 확률: $\frac{1}{2}$

**라운드 2** (A=5, B=4에서):
- A가 이기면 → A 최종 승리. 확률: $\frac{1}{2} \times \frac{1}{2} = \frac{1}{4}$
- B가 이기면 → A=5, B=5. 계속 진행. 확률: $\frac{1}{2} \times \frac{1}{2} = \frac{1}{4}$

**라운드 3** (A=5, B=5에서):
- A가 이기면 → A 최종 승리. 확률: $\frac{1}{4} \times \frac{1}{2} = \frac{1}{8}$
- B가 이기면 → B 최종 승리. 확률: $\frac{1}{4} \times \frac{1}{2} = \frac{1}{8}$

### 결과

$$P(\text{A 승리}) = \frac{1}{2} + \frac{1}{4} + \frac{1}{8} = \frac{4+2+1}{8} = \frac{7}{8}$$

$$P(\text{B 승리}) = \frac{1}{8}$$

**검증**: $\frac{7}{8} + \frac{1}{8} = 1$ (전체 확률 = 1)

In [ ]:
# Pascal의 풀이: 재귀적 확률 계산
def pascal_probability(a_wins, b_wins, target=6, p=0.5):
    """A가 최종 승리할 확률을 재귀적으로 계산"""
    if a_wins >= target:
        return 1.0  # A가 이미 승리
    if b_wins >= target:
        return 0.0  # B가 이미 승리
    # A가 이 라운드 이기는 경우 + B가 이 라운드 이기는 경우
    return p * pascal_probability(a_wins + 1, b_wins, target, p) + \
           (1 - p) * pascal_probability(a_wins, b_wins + 1, target, p)

P_A_wins = pascal_probability(5, 3)
P_B_wins = 1 - P_A_wins

total_prize = 100_000_000  # 1억 원

print("=" * 60)
print("Pascal의 풀이: 의사결정 트리")
print("=" * 60)
print(f"현재 점수: A={5}승, B={3}승 (6승 필요)")
print(f"\nA가 최종 승리할 확률: P(A) = {P_A_wins:.4f} = 7/8")
print(f"B가 최종 승리할 확률: P(B) = {P_B_wins:.4f} = 1/8")
print(f"확률 합계: {P_A_wins + P_B_wins:.4f} (검증)")
print(f"\n공평한 상금 분배:")
print(f"  A의 몫: {total_prize * P_A_wins:,.0f}원 ({P_A_wins*100:.1f}%)")
print(f"  B의 몫: {total_prize * P_B_wins:,.0f}원 ({P_B_wins*100:.1f}%)")

In [ ]:
# 의사결정 트리 시각화
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 9)
ax.axis('off')
ax.set_title('Pascal의 의사결정 트리 (Decision Tree)\n현재: A=5승, B=3승', 
             fontsize=18, fontweight='bold', pad=20)

# 노드 정의: (x, y, label, type)
# type: 'start', 'A_win', 'B_win', 'continue'
nodes = {
    'start': (1, 5, 'A:5, B:3', 'start'),
    'r1_a': (4, 7, 'A 승리!\n(6승 달성)', 'A_win'),
    'r1_b': (4, 3, 'A:5, B:4', 'continue'),
    'r2_a': (7, 5, 'A 승리!\n(6승 달성)', 'A_win'),
    'r2_b': (7, 1, 'A:5, B:5', 'continue'),
    'r3_a': (10, 2.5, 'A 승리!\n(6승 달성)', 'A_win'),
    'r3_b': (10, -0.5, 'B 승리!\n(6승 달성)', 'B_win'),
}

# 색상 설정
color_map = {'start': '#3498db', 'A_win': '#2ecc71', 'B_win': '#e74c3c', 'continue': '#f39c12'}

# 노드 그리기
for name, (x, y, label, ntype) in nodes.items():
    bbox = dict(boxstyle='round,pad=0.5', facecolor=color_map[ntype], alpha=0.8, edgecolor='black')
    ax.text(x, y, label, ha='center', va='center', fontsize=11, fontweight='bold',
            bbox=bbox, color='white' if ntype != 'continue' else 'black')

# 간선 그리기 (화살표 + 확률 레이블)
edges = [
    ('start', 'r1_a', 'A 이김\np=1/2', 'up'),
    ('start', 'r1_b', 'B 이김\np=1/2', 'down'),
    ('r1_b', 'r2_a', 'A 이김\np=1/2', 'up'),
    ('r1_b', 'r2_b', 'B 이김\np=1/2', 'down'),
    ('r2_b', 'r3_a', 'A 이김\np=1/2', 'up'),
    ('r2_b', 'r3_b', 'B 이김\np=1/2', 'down'),
]

for src, dst, label, direction in edges:
    sx, sy = nodes[src][0], nodes[src][1]
    dx, dy = nodes[dst][0], nodes[dst][1]
    color = '#2ecc71' if 'A 이김' in label else '#e74c3c'
    ax.annotate('', xy=(dx-0.7, dy), xytext=(sx+0.7, sy),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    mid_x = (sx + dx) / 2
    mid_y = (sy + dy) / 2
    ax.text(mid_x, mid_y + (0.4 if direction == 'up' else -0.4), label,
            ha='center', va='center', fontsize=10, color=color, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=color))

# 확률 요약 텍스트
summary_text = (
    "A 승리 경로:\n"
    "  경로 1: A이김 → P = 1/2\n"
    "  경로 2: B이김→A이김 → P = 1/4\n"
    "  경로 3: B이김→B이김→A이김 → P = 1/8\n"
    "  합계: 1/2 + 1/4 + 1/8 = 7/8\n\n"
    "B 승리 경로:\n"
    "  경로 4: B이김→B이김→B이김 → P = 1/8"
)
ax.text(5.5, 8.5, summary_text, fontsize=10, fontfamily='Malgun Gothic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9, edgecolor='gray'),
        verticalalignment='top')

plt.tight_layout()
plt.show()

### 해석

Pascal의 의사결정 트리에서:
- A가 승리하는 **3가지 경로**가 존재하며, 각 경로의 확률을 합산하면 $\frac{7}{8}$이다.
- B가 승리하려면 **연속 3번** 이겨야 하며, 이 확률은 $\frac{1}{2} \times \frac{1}{2} \times \frac{1}{2} = \frac{1}{8}$이다.
- 각 경로의 확률은 **독립사건의 결합확률**(곱셈)으로 계산된다.
- 서로 다른 경로는 **배반사건**이므로 확률을 더할 수 있다(덧셈).

---

## 풀이 2: Fermat의 접근 — 모든 경우의 수 완전 열거

### 핵심 아이디어

Fermat는 다른 관점에서 접근했다:

> 게임이 실제로 중단되었더라도, **최대한의 남은 라운드를 모두 진행한다**고 가정하고, 모든 가능한 결과를 열거한다.

### 남은 라운드 수 계산

- A는 1승 필요, B는 3승 필요
- 최악의 경우 $1 + 3 - 1 = 3$ 라운드가 필요 (한쪽이 필요한 승수 도달 직전까지)
- 따라서 **3라운드를 모두 진행**한 결과를 열거한다.

### 3라운드의 모든 결과 ($2^3 = 8$가지)

각 라운드에서 A가 이기면 A, B가 이기면 B로 표기:

| 순번 | 결과 | A 추가승 | B 추가승 | 최종 승자 | 실제 종료 시점 |
|------|------|----------|----------|-----------|---------------|
| 1 | AAA | 3 | 0 | **A** (1라운드에서 이미 6승) | 1라운드 |
| 2 | AAB | 2 | 1 | **A** (1라운드에서 이미 6승) | 1라운드 |
| 3 | ABA | 2 | 1 | **A** (1라운드에서 이미 6승) | 1라운드 |
| 4 | ABB | 1 | 2 | **A** (1라운드에서 이미 6승) | 1라운드 |
| 5 | BAA | 2 | 1 | **A** (2라운드에서 6승) | 2라운드 |
| 6 | BAB | 1 | 2 | **A** (2라운드에서 6승) | 2라운드 |
| 7 | BBA | 1 | 2 | **A** (3라운드에서 6승) | 3라운드 |
| 8 | BBB | 0 | 3 | **B** (3라운드에서 6승) | 3라운드 |

### 결과

$$P(\text{A 승리}) = \frac{7}{8}, \quad P(\text{B 승리}) = \frac{1}{8}$$

Pascal의 결과와 정확히 일치한다!

In [ ]:
# Fermat의 풀이: 모든 경우의 수 완전 열거
print("=" * 70)
print("Fermat의 풀이: 3라운드 결과 완전 열거")
print("=" * 70)

# 현재 점수
a_current = 5
b_current = 3
target = 6

# 남은 최대 라운드 수
a_needs = target - a_current  # 1승 필요
b_needs = target - b_current  # 3승 필요
max_rounds = a_needs + b_needs - 1  # 3라운드

print(f"A 추가 필요 승수: {a_needs}")
print(f"B 추가 필요 승수: {b_needs}")
print(f"최대 남은 라운드: {max_rounds}")
print(f"총 경우의 수: 2^{max_rounds} = {2**max_rounds}가지")
print()

# 모든 경우의 수 열거
outcomes = list(product(['A', 'B'], repeat=max_rounds))

a_win_count = 0
b_win_count = 0

print(f"{'순번':>4} {'결과':>6} {'A추가승':>8} {'B추가승':>8} {'최종승자':>8} {'종료시점':>10}")
print("-" * 50)

for i, outcome in enumerate(outcomes, 1):
    a_extra = outcome.count('A')
    b_extra = outcome.count('B')
    
    # 실제 종료 시점 계산
    end_round = max_rounds
    a_running, b_running = a_current, b_current
    for r, result in enumerate(outcome, 1):
        if result == 'A':
            a_running += 1
        else:
            b_running += 1
        if a_running >= target or b_running >= target:
            end_round = r
            break
    
    # 승자 판정 (첫 번째로 target에 도달한 쪽)
    winner = 'A' if a_current + sum(1 for x in outcome[:end_round] if x == 'A') >= target else 'B'
    
    if winner == 'A':
        a_win_count += 1
    else:
        b_win_count += 1
    
    result_str = ''.join(outcome)
    print(f"{i:>4} {result_str:>6} {a_extra:>8} {b_extra:>8} {'  ★A':>8} {end_round:>8}라운드" 
          if winner == 'A' else 
          f"{i:>4} {result_str:>6} {a_extra:>8} {b_extra:>8} {'  ★B':>8} {end_round:>8}라운드")

print("-" * 50)
print(f"\nA 승리: {a_win_count}/{len(outcomes)} = {a_win_count/len(outcomes):.4f} = 7/8")
print(f"B 승리: {b_win_count}/{len(outcomes)} = {b_win_count/len(outcomes):.4f} = 1/8")

In [ ]:
# 두 풀이 방법 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 왼쪽: 파이 차트 — 확률 분배
labels = ['A의 승리 확률\n(7/8 = 87.5%)', 'B의 승리 확률\n(1/8 = 12.5%)']
sizes = [7/8, 1/8]
colors_pie = ['#2ecc71', '#e74c3c']
explode = (0.05, 0.05)

wedges, texts, autotexts = axes[0].pie(sizes, labels=labels, colors=colors_pie, explode=explode,
                                        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
for autotext in autotexts:
    autotext.set_fontsize(14)
    autotext.set_fontweight('bold')
axes[0].set_title('승리 확률 분배', fontsize=16, fontweight='bold')

# 오른쪽: 상금 분배 막대그래프
prize_a = total_prize * 7 / 8
prize_b = total_prize * 1 / 8

bars = axes[1].bar(['A', 'B'], [prize_a, prize_b], color=colors_pie, edgecolor='white', width=0.5)
axes[1].set_ylabel('상금 (원)', fontsize=14)
axes[1].set_title('공평한 상금 분배', fontsize=16, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

for bar, prize in zip(bars, [prize_a, prize_b]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1_000_000,
                f'{prize:,.0f}원', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 해석

- **공평한 분배**: A는 87,500,000원 (87.5%), B는 12,500,000원 (12.5%)을 받아야 한다.
- 이 분배는 "과거에 누가 더 많이 이겼나"가 아니라, **"앞으로 누가 최종 승리할 가능성이 높은가"**에 기반한다.
- 이 개념이 바로 **기대값(Expected Value)**의 원형이다:

$$E[\text{A의 상금}] = P(\text{A 승리}) \times \text{총 상금} = \frac{7}{8} \times 1\text{억} = 8750\text{만 원}$$

---

## 시뮬레이션 검증

100,000번의 게임 시뮬레이션으로 이론적 결과를 검증한다.

In [ ]:
# 시뮬레이션: 100,000번 게임 시뮬레이션
np.random.seed(42)
n_simulations = 100_000

a_start = 5
b_start = 3
target = 6

a_wins_sim = 0
b_wins_sim = 0
rounds_played = []  # 각 게임에서 진행된 라운드 수

for _ in range(n_simulations):
    a_score = a_start
    b_score = b_start
    rounds = 0
    
    while a_score < target and b_score < target:
        rounds += 1
        if np.random.random() < 0.5:
            a_score += 1
        else:
            b_score += 1
    
    rounds_played.append(rounds)
    if a_score >= target:
        a_wins_sim += 1
    else:
        b_wins_sim += 1

sim_P_A = a_wins_sim / n_simulations
sim_P_B = b_wins_sim / n_simulations

print("=" * 60)
print(f"시뮬레이션 결과 ({n_simulations:,}회 반복)")
print("=" * 60)
print(f"A 승리 횟수: {a_wins_sim:,} ({sim_P_A:.4f})")
print(f"B 승리 횟수: {b_wins_sim:,} ({sim_P_B:.4f})")
print(f"\n이론값과 비교:")
print(f"  P(A 승리): 이론 = {7/8:.4f}, 시뮬레이션 = {sim_P_A:.4f}, 오차 = {abs(7/8 - sim_P_A):.4f}")
print(f"  P(B 승리): 이론 = {1/8:.4f}, 시뮬레이션 = {sim_P_B:.4f}, 오차 = {abs(1/8 - sim_P_B):.4f}")
print(f"\n평균 진행 라운드 수: {np.mean(rounds_played):.2f}")
print(f"최대 진행 라운드 수: {max(rounds_played)}")

In [ ]:
# 시뮬레이션 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 이론값 vs 시뮬레이션 비교
categories = ['A 승리', 'B 승리']
theoretical = [7/8, 1/8]
simulated = [sim_P_A, sim_P_B]

x_pos = np.arange(len(categories))
width = 0.35

bars1 = axes[0].bar(x_pos - width/2, theoretical, width, label='이론값', color='#3498db', edgecolor='white')
bars2 = axes[0].bar(x_pos + width/2, simulated, width, label=f'시뮬레이션 (n={n_simulations:,})', 
                     color='#e67e22', edgecolor='white')

axes[0].set_ylabel('확률', fontsize=14)
axes[0].set_title('이론값 vs 시뮬레이션 결과', fontsize=16, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(categories, fontsize=13)
axes[0].legend(fontsize=12)
axes[0].set_ylim(0, 1)

for bar, val in zip(bars1, theoretical):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
for bar, val in zip(bars2, simulated):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

# 오른쪽: 큰 수의 법칙 — 누적 A 승률 수렴
# 시뮬레이션을 다시 돌려서 누적 확률 추적
np.random.seed(42)
cumulative_a_wins = 0
cum_probs = []

for i in range(n_simulations):
    a_s, b_s = a_start, b_start
    while a_s < target and b_s < target:
        if np.random.random() < 0.5:
            a_s += 1
        else:
            b_s += 1
    if a_s >= target:
        cumulative_a_wins += 1
    cum_probs.append(cumulative_a_wins / (i + 1))

axes[1].plot(range(1, n_simulations + 1), cum_probs, color='#8da0cb', alpha=0.8, linewidth=1)
axes[1].axhline(y=7/8, color='red', linestyle='--', linewidth=2, label='이론값 = 7/8 = 0.8750')
axes[1].set_xlabel('시뮬레이션 횟수', fontsize=12)
axes[1].set_ylabel('누적 A 승리 비율', fontsize=12)
axes[1].set_title('큰 수의 법칙: A 승리 확률의 수렴', fontsize=16, fontweight='bold')
axes[1].set_xscale('log')
axes[1].legend(fontsize=12)
axes[1].set_ylim(0.7, 1.0)

plt.tight_layout()
plt.show()

### 해석

- 100,000번 시뮬레이션 결과가 이론값($\frac{7}{8}$)과 매우 근접함을 확인하였다.
- 오른쪽 그래프에서 시행 횟수가 늘어남에 따라 누적 비율이 이론값으로 **수렴**하는 것을 볼 수 있다.
- 이는 **큰 수의 법칙**의 직관적 증거이다.

---

## 두 풀이의 비교

| 항목 | Pascal의 풀이 | Fermat의 풀이 |
|------|--------------|---------------|
| **접근 방식** | 의사결정 트리 (순차적 분석) | 모든 경우의 수 열거 (전체 공간 분석) |
| **핵심 도구** | 조건부 확률, 분기 분석 | 결합확률, 완전 열거 |
| **장점** | 게임 진행 과정을 직관적으로 보여줌 | 체계적이고 빠짐없는 열거 |
| **단점** | 남은 라운드가 많으면 트리가 복잡 | 경우의 수가 기하급수적 증가 가능 |
| **결과** | $P(A) = 7/8, P(B) = 1/8$ | $P(A) = 7/8, P(B) = 1/8$ |
| **현대적 확장** | 동적 프로그래밍, 게임 이론 | 조합론, 이항분포 |

### 이항분포를 이용한 일반화

B가 승리하려면 남은 3라운드에서 3번 모두 이겨야 한다:

$$P(\text{B 승리}) = \binom{3}{3} \left(\frac{1}{2}\right)^3 = \frac{1}{8}$$

일반적으로, A가 $a$승 더 필요하고 B가 $b$승 더 필요할 때:

$$P(\text{B 승리}) = \sum_{k=b}^{a+b-1} \binom{a+b-1}{k} p^k (1-p)^{a+b-1-k}$$

여기서 $p = 0.5$이고, 이 문제에서 $a=1, b=3$이다.

In [ ]:
# 이항분포를 이용한 일반화 검증
from scipy.stats import binom

a_needs = 1  # A가 더 이겨야 할 횟수
b_needs = 3  # B가 더 이겨야 할 횟수
total_remaining = a_needs + b_needs - 1  # 최대 남은 라운드

# B가 승리할 확률: 남은 라운드에서 B가 b_needs 이상 이길 확률
P_B_binom = sum(binom.pmf(k, total_remaining, 0.5) for k in range(b_needs, total_remaining + 1))
P_A_binom = 1 - P_B_binom

print("=" * 50)
print("이항분포를 이용한 일반화")
print("=" * 50)
print(f"A 추가 필요 승수: {a_needs}")
print(f"B 추가 필요 승수: {b_needs}")
print(f"최대 남은 라운드: {total_remaining}")
print(f"\nP(A 승리) = {P_A_binom:.4f} = 7/8")
print(f"P(B 승리) = {P_B_binom:.4f} = 1/8")
print(f"\n공평한 상금 분배:")
print(f"  A: {total_prize * P_A_binom:,.0f}원")
print(f"  B: {total_prize * P_B_binom:,.0f}원")

---

## 종합 정리

### 문제의 답

| 항목 | 값 |
|------|----|
| A의 최종 승리 확률 | $\frac{7}{8} = 87.5\%$ |
| B의 최종 승리 확률 | $\frac{1}{8} = 12.5\%$ |
| A의 공평한 몫 | 87,500,000원 |
| B의 공평한 몫 | 12,500,000원 |

### 핵심 개념 요약

1. **결합확률**: 독립사건의 결합확률은 개별 확률의 곱 → $P(BBB) = \frac{1}{2} \times \frac{1}{2} \times \frac{1}{2} = \frac{1}{8}$
2. **배반사건의 덧셈**: 서로 다른 경로의 확률은 더할 수 있다 → $P(A) = \frac{1}{2} + \frac{1}{4} + \frac{1}{8} = \frac{7}{8}$
3. **기대값**: 미래의 가능성에 기반한 공평한 가치 산정 → $E = P \times \text{상금}$
4. **역사적 의의**: 이 문제가 확률론, 보험수학, 금융공학의 시초가 되었다